In [2]:
import time
import numpy as np

# RA (HMS) y Dec (DMS)
bbs_raw = np.loadtxt("J_MNRAS_384_775/table2.dat", usecols=range(1, 7))
n_bss = bbs_raw.shape[0]
bss_cat = np.zeros((n_bss, 3))

for i in range(n_bss):
    obj_id = float(i + 1)
    h = bbs_raw[i, 0]
    m = bbs_raw[i, 1]
    s = bbs_raw[i, 2]
    ra_deg = 15.0 * (h + (m / 60.0) + (s / 3600.0))

    d = bbs_raw[i, 3]
    dm = bbs_raw[i, 4]
    ds = bbs_raw[i, 5]
    if d >= 0:
        dec_deg = d + (dm / 60.0) + (ds / 3600.0)
    else:
        dec_deg = -1.0 * (-d + (dm / 60.0) + (ds / 3600.0))

    bss_cat[i, 0] = obj_id
    bss_cat[i, 1] = ra_deg
    bss_cat[i, 2] = dec_deg

print("BSS cargado:", bss_cat.shape)

BSS cargado: (320, 3)


In [3]:
# RA y Dec en grados decimales
super_raw = np.loadtxt("super.csv", delimiter=",", skiprows=1, usecols=[0, 1])
n_super = super_raw.shape[0]
super_cat = np.zeros((n_super, 3))

for j in range(n_super):
    obj_id = float(j + 1)
    ra_deg = super_raw[j, 0]
    dec_deg = super_raw[j, 1]
    super_cat[j, 0] = obj_id
    super_cat[j, 1] = ra_deg
    super_cat[j, 2] = dec_deg

print("SuperCOSMOS cargado:", super_cat.shape)

SuperCOSMOS cargado: (500, 3)


In [4]:
def ang_dist(ra1, dec1, ra2, dec2):
    # Conversion de grados a radianes
    ra1_rad = np.radians(ra1)
    dec1_rad = np.radians(dec1)
    ra2_rad = np.radians(ra2)
    dec2_rad = np.radians(dec2)

    # Formula de Haversine
    dif_dec = np.abs(dec1_rad - dec2_rad) / 2.0
    dif_ra = np.abs(ra1_rad - ra2_rad) / 2.0
    a = np.sin(dif_dec) ** 2
    b = np.cos(dec1_rad) * np.cos(dec2_rad) * (np.sin(dif_ra) ** 2)
    d_rad = 2.0 * np.arcsin(np.sqrt(a + b))
    d_deg = np.degrees(d_rad)
    return d_deg


def crossmatch(cat1, cat2, max_dist_arcsec):
    # Convertir distancia maxima de arcsegundos a grados
    max_dist_deg = max_dist_arcsec / 3600.0

    matches = []
    non_matches = []

    for i in range(len(cat1)):
        id1 = int(cat1[i, 0])
        ra1 = cat1[i, 1]
        dec1 = cat1[i, 2]

        min_dist_deg = np.inf
        best_id2 = None

        for j in range(len(cat2)):
            id2 = int(cat2[j, 0])
            ra2 = cat2[j, 1]
            dec2 = cat2[j, 2]
            dist_deg = ang_dist(ra1, dec1, ra2, dec2)

            if dist_deg < min_dist_deg:
                min_dist_deg = dist_deg
                best_id2 = id2

        min_dist_arcsec = min_dist_deg * 3600.0
        if min_dist_deg <= max_dist_deg:
            matches.append((id1, best_id2, min_dist_arcsec))
        else:
            non_matches.append(id1)

    # Ordenar ambas listas por el ID del primer catalogo
    matches.sort(key=lambda item: item[0])
    non_matches.sort()

    return matches, non_matches

In [5]:
def medir_tiempo_crossmatch(cat1, cat2, max_dist_arcsec):
    # Medicion de tiempo transcurrido en segundos
    t_inicio = time.perf_counter()
    matches, non_matches = crossmatch(cat1, cat2, max_dist_arcsec)
    t_fin = time.perf_counter()
    elapsed_sec = t_fin - t_inicio
    return matches, non_matches, elapsed_sec

In [6]:
# Distancia maxima: 10 arcsegundos
max_dist_10 = 10.0
matches_10, non_matches_10, tiempo_10 = medir_tiempo_crossmatch(bss_cat, super_cat, max_dist_10)

print("Tiempo (10 arcsec) [s]:", tiempo_10)
print("Total matches (10 arcsec):", len(matches_10))
print("Total non-matches (10 arcsec):", len(non_matches_10))
print("Primeros 5 matches:", matches_10[:5])
print("Primeros 10 non-matches:", non_matches_10[:10])

Tiempo (10 arcsec) [s]: 0.29237916900092387
Total matches (10 arcsec): 131
Total non-matches (10 arcsec): 189
Primeros 5 matches: [(1, 2, np.float64(0.39558999379356213)), (2, 4, np.float64(2.753944548207298)), (3, 5, np.float64(0.7510807033454759)), (4, 6, np.float64(0.4632227988150214)), (7, 11, np.float64(0.27598446324039627))]
Primeros 10 non-matches: [5, 6, 11, 14, 29, 45, 47, 52, 57, 58]
